# 04 - Rolling Portfolio Optimization

This notebook estimates factor weights from rolling historical windows and applies each weight to the following month.

## 1. Load data and settings

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
current_directory = Path.cwd()
repository_root = current_directory.parent if current_directory.name == "notebooks" else current_directory

if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from src import load_config, load_monthly_returns, plot_weights, walk_forward_backtest

In [ ]:
config = load_config(repository_root / "config" / "config.yaml")
monthly_returns = load_monthly_returns(repository_root / config["data_file"])

tables_directory = repository_root / "results" / "tables"
weights_directory = repository_root / "results" / "portfolio_weights"
figures_directory = repository_root / "results" / "figures"
tables_directory.mkdir(parents=True, exist_ok=True)
weights_directory.mkdir(parents=True, exist_ok=True)
figures_directory.mkdir(parents=True, exist_ok=True)

## 2. Historical walk-forward allocations

In [ ]:
methods = [
    "equal_weight",
    "inverse_volatility",
    "sample_gmv",
    "shrinkage_gmv",
    "erc"
]

backtests = {
    method: walk_forward_backtest(
        monthly_returns,
        method=method,
        lookback=config["lookback_months"],
        rebalance_months=config["rebalance_frequency_months"],
        lag=1,
        max_weight=config["maximum_weight"],
        transaction_cost_bps=config["transaction_cost_bps"],
        factor_columns=config["factor_columns"],
        market_column=config["benchmark"]
    )
    for method in methods
}

In [ ]:
walk_forward_returns = pd.DataFrame({
    method: backtests[method]["returns"]["net_return"]
    for method in methods
})
walk_forward_gross_returns = pd.DataFrame({
    method: backtests[method]["returns"]["gross_return"]
    for method in methods
})
walk_forward_turnover = pd.DataFrame({
    method: backtests[method]["returns"]["turnover"]
    for method in methods
})

walk_forward_returns["market"] = backtests[methods[0]]["returns"]["benchmark_return"]
walk_forward_gross_returns["market"] = walk_forward_returns["market"]

walk_forward_returns.head()

In [ ]:
for method in methods:
    target_weights = backtests[method]["target_weights"].dropna(how="all")
    assert target_weights.sum(axis=1).sub(1).abs().max() < 1e-9
    assert target_weights.min().min() >= -1e-10
    assert target_weights.max().max() <= config["maximum_weight"] + 1e-10
    assert (backtests[method]["returns"]["estimation_end"].dropna().index > backtests[method]["returns"]["estimation_end"].dropna()).all()

## 3. Save returns, turnover and weights

In [ ]:
walk_forward_returns.to_csv(tables_directory / "walk_forward_returns.csv")
walk_forward_gross_returns.to_csv(tables_directory / "walk_forward_gross_returns.csv")
walk_forward_turnover.to_csv(tables_directory / "walk_forward_turnover.csv")

for method in methods:
    backtests[method]["weights"].to_csv(weights_directory / f"{method}_weights.csv")
    backtests[method]["target_weights"].to_csv(weights_directory / f"{method}_target_weights.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_weights(backtests["shrinkage_gmv"]["weights"], ax=ax)
ax.set_title("Rolling shrinkage GMV weights")
plt.tight_layout()
fig.savefig(figures_directory / "rolling_shrinkage_gmv_weights.png", dpi=150)
plt.show()

## Conclusion

The saved return series use only information from completed months. Portfolio weights drift between the scheduled rebalancing dates, and costs are applied to drift-adjusted turnover.